# Building Custom Tools

This notebook accompanies [Agent Foundry](https://agent-foundry-pi.vercel.app) — CrewAI roadmap: custom tools with `@tool` and `BaseTool`.

In [ ]:
!pip install -q crewai crewai-tools pydantic

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## Custom tools: `@tool` and `BaseTool`

Below: a quick **decorator** tool, a **class** tool with Pydantic `args_schema`, an **agent** that receives both, a **task** that asks the model to use them, and a **crew** run via `kickoff()`.

In [ ]:
from crewai import Agent, Crew, Task
from crewai.tools import BaseTool, tool
from pydantic import BaseModel, Field


@tool("Calculate Price")
def calculate_price(quantity: int, unit_price: float) -> str:
    """Calculate total price for a given quantity and unit price."""
    total = quantity * unit_price
    return f"Total price: ${total:.2f}"


class SearchInput(BaseModel):
    query: str = Field(..., description="The search query")
    max_results: int = Field(default=5, description="Maximum results to return")


class DatabaseSearchTool(BaseTool):
    name: str = "Database Search"
    description: str = "Search the internal database for records"
    args_schema: type[BaseModel] = SearchInput

    def _run(self, query: str, max_results: int = 5) -> str:
        return f"Found results for: {query} (limit {max_results})"


db_search = DatabaseSearchTool()

analyst = Agent(
    role="Operations Analyst",
    goal="Use the pricing and database tools to answer the task; do not invent tool outputs.",
    backstory="You always call tools when numbers or lookups are needed.",
    tools=[calculate_price, db_search],
    verbose=True,
)

task = Task(
    description=(
        "Use Calculate Price for quantity 4 and unit price 12.5. "
        "Then use Database Search with query 'inventory' and max_results 3. "
        "Summarize both tool outputs in plain English."
    ),
    expected_output="A brief summary of the pricing result and the database search result.",
    agent=analyst,
)

crew = Crew(agents=[analyst], tasks=[task], verbose=True)
result = crew.kickoff()
print(result)

## Key takeaways

- Custom tools connect agents to proprietary APIs, databases, and your own logic.
- **`@tool`** is fastest for small functions; **`BaseTool` + Pydantic** gives structured, validated arguments.
- Pass tools with `Agent(..., tools=[...])`; the LLM picks which tool to call during `crew.kickoff()`.
- Tools need a clear **name**, **description**, **type-annotated** parameters, and a **string** return value.